# 7. Tune (rule-based)

Grid-searches `top_k` against `ground_truth` and recommends the best value
(see `../docs/TUNING.md`). Seed ground truth first — this notebook includes a
demo seed for `three_level` from the synthetic invoices' own categories; for
real data call `import_ground_truth` with the team's classified export.

**Not part of the scheduled pipeline** — run interactively while tuning.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openai openpyxl pydantic 'databricks-sdk>=0.50.0' psycopg2-binary
%restart_python

In [ ]:
import os, sys, uuid
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import get_schema
from src.feedback import import_ground_truth
from src.tuner import tune_top_k
from src.llm import gateway_client

spark = get_spark()
config = load_config()
client = gateway_client()   # Unity Gateway

dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('lakebase_instance', config.lakebase_instance)
dbutils.widgets.text('llm_endpoint', config.large_llm_endpoint)
dbutils.widgets.text('embed_endpoint', 'databricks-gte-large-en')
dbutils.widgets.text('tune_schema', 'three_level')
dbutils.widgets.text('top_k_grid', '10,20,30,50')  # comma-separated values to try
dbutils.widgets.text('sample_size', '100')         # labelled invoices to score each on
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
lakebase_instance = dbutils.widgets.get('lakebase_instance')
llm_endpoint = dbutils.widgets.get('llm_endpoint')
embed_endpoint = dbutils.widgets.get('embed_endpoint')
tune_schema = dbutils.widgets.get('tune_schema')
top_k_grid = tuple(int(x) for x in dbutils.widgets.get('top_k_grid').split(',') if x.strip())
sample_size = int(dbutils.widgets.get('sample_size'))
spark.sql(f'USE {catalog}.{schema}')

## Lakebase connection

The tuner retrieves candidates from the pgvector `examples` table.

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg2

w = WorkspaceClient()

def pg_connect():
    inst = w.database.get_database_instance(name=lakebase_instance)
    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()), instance_names=[lakebase_instance]
    )
    me = w.current_user.me()
    user = me.emails[0].value if me.emails else me.user_name
    return psycopg2.connect(
        host=inst.read_write_dns, dbname=config.lakebase_dbname,
        user=user, password=cred.token, sslmode='require',
    )

## (Demo) Seed ground truth for three_level

The generated invoices carry their true categories, so we can seed
`ground_truth` to demonstrate the loop end to end. For real data, replace
this with `import_ground_truth(..., labeled=<your classified export>)`.

In [ ]:
if tune_schema == 'three_level':
    seed = spark.sql(f'''
        SELECT order_id, category_level_3 AS code
        FROM {catalog}.{schema}.invoices
        WHERE category_level_3 IS NOT NULL
    ''').toPandas()
    import_ground_truth(spark, catalog=catalog, schema_db=schema,
                        schema_name='three_level', labeled=seed, labeled_by='demo_seed')
else:
    print('Demo seed is wired for three_level only; import your own ground truth for other schemas.')

## Tune top_k against ground truth

Re-embed (notebook 2) after seeding if you want the new ground-truth rows as
few-shot examples. Then grid-search `top_k`; apply the recommendation by
setting `top_k` on the schema's `SchemaSpec` and re-running notebook 3.

In [ ]:
results = tune_top_k(
    spark, get_schema(tune_schema),
    catalog=catalog, schema_db=schema,
    pg_connect=pg_connect, client=client,
    chat_endpoint=llm_endpoint, embed_endpoint=embed_endpoint,
    grid=top_k_grid, sample_size=sample_size,
)